# 🔤 Pelatihan Custom WordPiece Tokenizer URL — Metode URLBERT

**Notebook**: `01_train_custom_url_tokenizer_colab.ipynb`  
**Lingkungan**: Google Colab (CPU — tidak memerlukan GPU)  
**Tujuan**: Melatih tokenizer WordPiece baru **dari nol** (*training from scratch*) menggunakan corpus URL dari dataset `vonDataset_clean.csv`, mengadopsi rancangan **URL-Tokenizer** dari paper **URLBERT** (Li et al., 2024/2025).

### Keluaran Utama
| Artefak | Lokasi (Google Drive) |
|---|---|
| `vocab.txt` | `custom_url_tokenizer/vocab.txt` |
| `tokenizer.json` | `custom_url_tokenizer/tokenizer.json` |
| Log Progres | `SKRIPSI_LOG.md` (append otomatis) |

### Perbedaan dengan Tokenizer Default BERT
| Aspek | Default BERT | URL-Tokenizer (URLBERT) |
|---|---|---|
| Corpus pelatihan | Wikipedia + BookCorpus (bahasa natural) | Corpus URL (domain, path, query string) |
| Token `[SEP]` | Ada (pemisah kalimat) | **Ditiadakan** (tidak ada ketergantungan antarkalimat) |
| Token tambahan | — | `[REP]` (replacement), `[SHU]` (shuffle) |
| Kosakata | Kata-kata bahasa natural | Subword khusus pola URL (`.com`, `login`, `//`, dsb.) |

**Referensi**: Li et al., *URLBERT: A Contrastive and Adversarial Pre-trained Model for URL Classification*, 2024/2025.

---
## Section 1 — Setup Google Drive & Dependensi

Mounting Google Drive agar seluruh artefak tersimpan permanen dan tidak hilang saat runtime Colab terputus, serta instalasi pustaka yang diperlukan.

In [1]:
# ============================================================
# 1a. Mounting Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print("[INFO] ✅ Google Drive berhasil di-mount pada /content/drive")

Mounted at /content/drive
[INFO] ✅ Google Drive berhasil di-mount pada /content/drive


In [2]:
# ============================================================
# 1b. Instalasi Library Pendukung
# ============================================================
!pip install -q pandas tokenizers transformers
print("\n[INFO] ✅ Seluruh dependensi berhasil diinstal.")


[INFO] ✅ Seluruh dependensi berhasil diinstal.


---
## Section 2 — Konfigurasi Path Proyek (Terpusat & Fleksibel)

Seluruh path I/O didefinisikan di satu tempat agar mudah diubah.  
Notebook akan **mencari file `vonDataset_clean.csv` secara otomatis** di dalam folder proyek jika path yang ditentukan tidak ditemukan.

> ⚠️ **Sesuaikan variabel `PROJECT_DIR`** di bawah ini dengan lokasi folder skripsi Anda di Google Drive.

In [3]:
# ============================================================
# 2. Konfigurasi Path Proyek
# ============================================================
import os

# ┌──────────────────────────────────────────────────────────┐
# │  ⚠️ SESUAIKAN PATH INI DENGAN FOLDER ANDA DI GDRIVE    │
# └──────────────────────────────────────────────────────────┘
PROJECT_DIR = "/content/drive/MyDrive/Skripsi"

# --- Path Turunan (Otomatis) ---
DATASET_FILENAME     = "vonDataset_clean.csv"
DATASET_PATH         = os.path.join(PROJECT_DIR, "new data", DATASET_FILENAME)
TOKENIZER_OUTPUT_DIR = os.path.join(PROJECT_DIR, "custom_url_tokenizer")
LOG_PATH             = os.path.join(PROJECT_DIR, "SKRIPSI_LOG.md")

# File sementara di penyimpanan lokal Colab (lebih cepat untuk I/O)
CORPUS_TEMP_FILE = "/content/temp_corpus_urls.txt"

# Variabel opsional: set manual jika nama kolom URL bukan 'url'
URL_COLUMN_NAME = None  # Contoh: "URL", "url", "urls"

# ============================================================
# Verifikasi & Pencarian File Otomatis
# ============================================================

# 1. Cek PROJECT_DIR
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"❌ Direktori proyek tidak ditemukan: '{PROJECT_DIR}'\n"
        f"   Pastikan Google Drive sudah di-mount dan path sudah benar."
    )
print(f"[INFO] ✅ Direktori proyek ditemukan: '{PROJECT_DIR}'")

# 2. Cek DATASET_PATH — jika tidak ditemukan, cari otomatis di seluruh PROJECT_DIR
if not os.path.isfile(DATASET_PATH):
    print(f"[WARN] ⚠️ File tidak ditemukan di path utama:")
    print(f"       '{DATASET_PATH}'")
    print(f"[INFO] 🔍 Mencari '{DATASET_FILENAME}' secara rekursif di '{PROJECT_DIR}'...")
    print()

    found_files = []
    for root, dirs, files in os.walk(PROJECT_DIR):
        for fname in files:
            if fname.lower() == DATASET_FILENAME.lower():
                full = os.path.join(root, fname)
                size_mb = os.path.getsize(full) / (1024 * 1024)
                found_files.append((full, size_mb))

    if found_files:
        print(f"[INFO] Ditemukan {len(found_files)} file cocok:")
        for i, (fp, sz) in enumerate(found_files, 1):
            print(f"  [{i}] {fp} ({sz:.1f} MB)")

        # Ambil file terbesar (kemungkinan besar dataset utama)
        found_files.sort(key=lambda x: x[1], reverse=True)
        DATASET_PATH = found_files[0][0]
        print(f"\n[INFO] ✅ Menggunakan file terbesar: '{DATASET_PATH}'")
    else:
        # Tampilkan isi direktori untuk membantu debugging
        print(f"[ERROR] ❌ '{DATASET_FILENAME}' tidak ditemukan di manapun dalam '{PROJECT_DIR}'.")
        print(f"\n[DEBUG] Isi direktori '{PROJECT_DIR}':")
        for item in sorted(os.listdir(PROJECT_DIR)):
            item_path = os.path.join(PROJECT_DIR, item)
            if os.path.isdir(item_path):
                sub_items = os.listdir(item_path)
                print(f"  📁 {item}/ ({len(sub_items)} item)")
                for si in sorted(sub_items)[:10]:
                    si_path = os.path.join(item_path, si)
                    if os.path.isfile(si_path):
                        si_sz = os.path.getsize(si_path) / (1024*1024)
                        print(f"       └─ {si} ({si_sz:.1f} MB)")
                    else:
                        print(f"       └─ 📁 {si}/")
            else:
                sz = os.path.getsize(item_path) / (1024*1024)
                print(f"  📄 {item} ({sz:.1f} MB)")
        raise FileNotFoundError(
            f"\n❌ File '{DATASET_FILENAME}' tidak ditemukan.\n"
            f"   Upload file tersebut ke salah satu folder di '{PROJECT_DIR}' dan jalankan ulang cell ini."
        )
else:
    dataset_size_mb = os.path.getsize(DATASET_PATH) / (1024 * 1024)
    print(f"[INFO] ✅ File dataset ditemukan: '{DATASET_PATH}' ({dataset_size_mb:.1f} MB)")

# --- Ringkasan Konfigurasi ---
print()
print("[CONFIG] Konfigurasi path final:")
print(f"  PROJECT_DIR          : {PROJECT_DIR}")
print(f"  DATASET_PATH         : {DATASET_PATH}")
print(f"  TOKENIZER_OUTPUT_DIR : {TOKENIZER_OUTPUT_DIR}")
print(f"  LOG_PATH             : {LOG_PATH}")
print(f"  CORPUS_TEMP_FILE     : {CORPUS_TEMP_FILE}")

[INFO] ✅ Direktori proyek ditemukan: '/content/drive/MyDrive/Skripsi'
[INFO] ✅ File dataset ditemukan: '/content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv' (122.6 MB)

[CONFIG] Konfigurasi path final:
  PROJECT_DIR          : /content/drive/MyDrive/Skripsi
  DATASET_PATH         : /content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv
  TOKENIZER_OUTPUT_DIR : /content/drive/MyDrive/Skripsi/custom_url_tokenizer
  LOG_PATH             : /content/drive/MyDrive/Skripsi/SKRIPSI_LOG.md
  CORPUS_TEMP_FILE     : /content/temp_corpus_urls.txt


---
## Section 3 — Pemuatan & Prapemrosesan Dataset URL

Langkah-langkah:
1. Muat CSV dari Google Drive menggunakan `pandas`.
2. Deteksi kolom URL secara otomatis (fleksibel terhadap variasi nama kolom).
3. Bersihkan data `NaN`/kosong dan terapkan **lowercasing** pada seluruh URL.
4. Tulis corpus bersih ke file teks sementara di penyimpanan lokal Colab untuk pelatihan tokenizer.

In [4]:
# ============================================================
# 3. Pemuatan & Prapemrosesan Dataset URL
# ============================================================
import pandas as pd

# --- Muat Dataset ---
print(f"[INFO] Memuat dataset dari Google Drive...")
print(f"  Path: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
print(f"[INFO] Dataset dimuat: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"[INFO] Kolom tersedia: {list(df.columns)}")
print()

# --- Deteksi Kolom URL secara Fleksibel ---
if URL_COLUMN_NAME is not None:
    url_col = URL_COLUMN_NAME
    if url_col not in df.columns:
        raise KeyError(
            f"❌ Kolom '{url_col}' tidak ditemukan di dataset.\n"
            f"   Kolom tersedia: {list(df.columns)}"
        )
else:
    # Strategi 1: Cari kolom bernama 'url' (case-insensitive)
    url_candidates = [c for c in df.columns if c.strip().lower() == "url"]
    if url_candidates:
        url_col = url_candidates[0]
    else:
        # Strategi 2: Cari kolom string pertama yang mengandung pola URL
        url_col = None
        for c in df.columns:
            if df[c].dtype == object:
                sample = df[c].dropna().head(100)
                if sample.str.contains(r"https?://", regex=True).mean() > 0.5:
                    url_col = c
                    break
        if url_col is None:
            raise ValueError(
                f"❌ Tidak dapat mendeteksi kolom URL secara otomatis.\n"
                f"   Kolom yang tersedia: {list(df.columns)}\n"
                f"   Silakan set variabel URL_COLUMN_NAME secara manual di Section 2."
            )

print(f"[INFO] Kolom URL terdeteksi: '{url_col}'")

# --- Pembersihan Data ---
urls_raw_count = len(df)
df = df.dropna(subset=[url_col])                          # Hapus NaN
df = df[df[url_col].astype(str).str.strip().ne("")]       # Hapus string kosong
urls_clean_count = len(df)
dropped_count = urls_raw_count - urls_clean_count

print(f"[INFO] Baris dihapus (NaN/kosong): {dropped_count:,}")
print(f"[INFO] Total URL bersih tersisa : {urls_clean_count:,}")

# --- Lowercasing (Case-Insensitive sesuai URLBERT) ---
urls_clean = df[url_col].astype(str).str.lower().tolist()
print(f"[INFO] Lowercasing diterapkan pada seluruh {len(urls_clean):,} URL.")
print()

# --- Ekspor ke File Teks Sementara (Lokal Colab, I/O lebih cepat) ---
with open(CORPUS_TEMP_FILE, "w", encoding="utf-8") as f:
    for url in urls_clean:
        f.write(url + "\n")

temp_size_mb = os.path.getsize(CORPUS_TEMP_FILE) / (1024 * 1024)
print(f"[INFO] Corpus sementara disimpan ke: '{CORPUS_TEMP_FILE}' ({temp_size_mb:.2f} MB)")
print(f"[INFO] Contoh URL bersih (5 pertama):")
for i, u in enumerate(urls_clean[:5], 1):
    print(f"  {i}. {u}")

[INFO] Memuat dataset dari Google Drive...
  Path: /content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv
[INFO] Dataset dimuat: 1,559,362 baris × 3 kolom
[INFO] Kolom tersedia: ['url', 'label', 'split']

[INFO] Kolom URL terdeteksi: 'url'
[INFO] Baris dihapus (NaN/kosong): 0
[INFO] Total URL bersih tersisa : 1,559,362
[INFO] Lowercasing diterapkan pada seluruh 1,559,362 URL.

[INFO] Corpus sementara disimpan ke: '/content/temp_corpus_urls.txt' (111.17 MB)
[INFO] Contoh URL bersih (5 pertama):
  1. http://bs-369.com/views/account/login.aspx?returnurl=/
  2. http://web.furell.net/web/login.html
  3. http://amelihair.ru/images/6550054213/745353769679/
  4. https://www.behance.net/gallery/23251525/self-branding
  5. http://ssl32.sslbr.co.vu/gol/


---
## Section 4 — Pelatihan WordPiece Tokenizer dari Nol (Spesifikasi URLBERT)

Menggunakan `BertWordPieceTokenizer` dari pustaka `tokenizers` (Hugging Face) dengan konfigurasi yang mengadopsi rancangan URL-Tokenizer pada paper **URLBERT**:

| Parameter | Nilai | Keterangan |
|---|---|---|
| `vocab_size` | **30.000** | Ukuran kosakata target |
| `special_tokens` | `[PAD]`, `[UNK]`, `[CLS]`, `[MASK]`, `[REP]`, `[SHU]` | Token khusus |
| `[SEP]` | **Ditiadakan** | Tidak ada ketergantungan antarkalimat pada URL |
| `lowercase` | `True` | Konsisten dengan prapemrosesan lowercasing |
| `strip_accents` | `False` | Mempertahankan karakter aksen asli |
| `clean_text` | `True` | Bersihkan karakter kontrol/whitespace |
| `min_frequency` | `2` | Token harus muncul minimal 2 kali |

In [5]:
# ============================================================
# 4. Pelatihan WordPiece Tokenizer dari Nol (URLBERT)
# ============================================================
import time
from tokenizers import BertWordPieceTokenizer

# --- Konfigurasi Tokenizer ---
VOCAB_SIZE     = 30_000
MIN_FREQUENCY  = 2
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[MASK]", "[REP]", "[SHU]"]

# --- Inisialisasi BertWordPieceTokenizer ---
tokenizer = BertWordPieceTokenizer(
    lowercase=True,
    strip_accents=False,
    clean_text=True
)

print("[INFO] Konfigurasi pelatihan tokenizer:")
print(f"  Algoritma       : WordPiece (BertWordPieceTokenizer)")
print(f"  vocab_size      : {VOCAB_SIZE:,}")
print(f"  special_tokens  : {SPECIAL_TOKENS}")
print(f"  min_frequency   : {MIN_FREQUENCY}")
print(f"  lowercase       : True")
print(f"  strip_accents   : False")
print(f"  clean_text      : True")
print(f"  corpus file     : {CORPUS_TEMP_FILE}")
print(f"  output dir      : {TOKENIZER_OUTPUT_DIR}")
print()
print("[INFO] ⏳ Memulai pelatihan tokenizer... (ini mungkin memakan beberapa menit)")

# --- Pelatihan ---
train_start = time.perf_counter()

tokenizer.train(
    files=[CORPUS_TEMP_FILE],
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    min_frequency=MIN_FREQUENCY
)

train_duration = time.perf_counter() - train_start
train_duration_str = f"{train_duration:.2f} detik"
if train_duration >= 60:
    mins = int(train_duration // 60)
    secs = train_duration % 60
    train_duration_str = f"{mins} menit {secs:.1f} detik ({train_duration:.2f}s)"

print(f"[INFO] ✅ Pelatihan tokenizer selesai dalam {train_duration_str}.")
print()

# --- Simpan ke Google Drive ---
os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)

# Simpan vocab.txt
tokenizer.save_model(TOKENIZER_OUTPUT_DIR)

# Simpan tokenizer.json (untuk kompatibilitas penuh dengan BertTokenizerFast)
tokenizer.save(os.path.join(TOKENIZER_OUTPUT_DIR, "tokenizer.json"))

# --- Verifikasi File Output ---
print(f"[INFO] Artefak tokenizer disimpan ke Google Drive:")
print(f"  Direktori: {TOKENIZER_OUTPUT_DIR}/")
for fname in sorted(os.listdir(TOKENIZER_OUTPUT_DIR)):
    fpath = os.path.join(TOKENIZER_OUTPUT_DIR, fname)
    fsize_kb = os.path.getsize(fpath) / 1024
    print(f"    → {fname} ({fsize_kb:.1f} KB)")

# --- Hitung Ukuran Kosakata Aktual ---
vocab_path = os.path.join(TOKENIZER_OUTPUT_DIR, "vocab.txt")
with open(vocab_path, "r", encoding="utf-8") as vf:
    actual_vocab_size = sum(1 for _ in vf)
print(f"\n[INFO] Ukuran kosakata aktual (vocab.txt): {actual_vocab_size:,} token")

# --- Hapus File Corpus Sementara ---
if os.path.exists(CORPUS_TEMP_FILE):
    os.remove(CORPUS_TEMP_FILE)
    print(f"[INFO] 🗑️ File sementara '{CORPUS_TEMP_FILE}' telah dihapus.")

[INFO] Konfigurasi pelatihan tokenizer:
  Algoritma       : WordPiece (BertWordPieceTokenizer)
  vocab_size      : 30,000
  special_tokens  : ['[PAD]', '[UNK]', '[CLS]', '[MASK]', '[REP]', '[SHU]']
  min_frequency   : 2
  lowercase       : True
  strip_accents   : False
  clean_text      : True
  corpus file     : /content/temp_corpus_urls.txt
  output dir      : /content/drive/MyDrive/Skripsi/custom_url_tokenizer

[INFO] ⏳ Memulai pelatihan tokenizer... (ini mungkin memakan beberapa menit)
[INFO] ✅ Pelatihan tokenizer selesai dalam 1 menit 29.0 detik (88.99s).

[INFO] Artefak tokenizer disimpan ke Google Drive:
  Direktori: /content/drive/MyDrive/Skripsi/custom_url_tokenizer/
    → tokenizer.json (676.3 KB)
    → vocab.txt (216.8 KB)

[INFO] Ukuran kosakata aktual (vocab.txt): 30,000 token
[INFO] 🗑️ File sementara '/content/temp_corpus_urls.txt' telah dihapus.


---
## Section 5 — Verifikasi & Uji Tokenisasi

Memuat kembali tokenizer yang telah dilatih menggunakan `BertTokenizerFast` dari pustaka `transformers` dan menguji tokenisasi pada beberapa contoh URL phishing untuk memastikan `vocab.txt` bekerja memecah domain, path, dan parameter query secara benar.

In [6]:
# ============================================================
# 5. Verifikasi & Uji Tokenisasi
# ============================================================
from transformers import BertTokenizerFast

# --- Muat Tokenizer dari Google Drive ---
print(f"[INFO] Memuat tokenizer dari: '{TOKENIZER_OUTPUT_DIR}/'")
loaded_tokenizer = BertTokenizerFast.from_pretrained(
    TOKENIZER_OUTPUT_DIR,
    do_lower_case=True
)

print(f"[INFO] ✅ Tokenizer berhasil dimuat.")
print(f"  Vocab size      : {loaded_tokenizer.vocab_size:,}")
print(f"  Special tokens  : {loaded_tokenizer.all_special_tokens}")
print(f"  Model max length: {loaded_tokenizer.model_max_length:,}")
print()

# --- URL Uji Utama (Phishing) ---
test_url_primary = "http://secure-login.bank-update.xyz/verify?id=123"

print("=" * 72)
print("  UJI TOKENISASI UTAMA")
print("=" * 72)
print(f"  URL : {test_url_primary}")
print()

encoding_primary = loaded_tokenizer(
    test_url_primary,
    add_special_tokens=True,
    return_tensors=None
)
tokens_primary = loaded_tokenizer.convert_ids_to_tokens(encoding_primary["input_ids"])

print(f"  Token subword ({len(tokens_primary)} token):")
print(f"    {tokens_primary}")
print(f"  Token IDs:")
print(f"    {encoding_primary['input_ids']}")
print(f"  Total panjang sekuens: {len(tokens_primary)}")
print()

# Simpan hasil untuk log
test_url_token_count = len(tokens_primary)

# --- URL Uji Tambahan ---
test_urls_extra = [
    "https://www.google.com/search?q=python+tutorial",
    "http://192.168.1.1/admin/login.php?redirect=home",
    "https://paypa1-secure.com.malicious.ru/signin",
    "http://amzn-verify.account-update.xyz/reset?token=abc123def",
    "https://www.github.com/huggingface/transformers",
]

print("=" * 72)
print("  UJI TOKENISASI TAMBAHAN (5 URL)")
print("=" * 72)

for i, url in enumerate(test_urls_extra, 1):
    enc = loaded_tokenizer(url, add_special_tokens=True)
    toks = loaded_tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"  [{i}] URL    : {url}")
    print(f"      Tokens : {toks}")
    print(f"      Length : {len(toks)} token")
    print(f"  {'-' * 68}")

print()
print("[INFO] ✅ Verifikasi tokenizer selesai — tokenizer berfungsi dengan benar.")

[INFO] Memuat tokenizer dari: '/content/drive/MyDrive/Skripsi/custom_url_tokenizer/'
[INFO] ✅ Tokenizer berhasil dimuat.
  Vocab size      : 30,000
  Special tokens  : ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
  Model max length: 1,000,000,000,000,000,019,884,624,838,656

  UJI TOKENISASI UTAMA
  URL : http://secure-login.bank-update.xyz/verify?id=123

  Token subword (21 token):
    ['[CLS]', 'http', ':', '/', '/', 'secure', '-', 'login', '.', 'bank', '-', 'update', '.', 'xyz', '/', 'verify', '?', 'id', '=', '123', '[SEP]']
  Token IDs:
    [2, 113, 30, 19, 19, 370, 17, 164, 18, 848, 17, 339, 18, 4004, 19, 935, 34, 286, 33, 2409, 30000]
  Total panjang sekuens: 21

  UJI TOKENISASI TAMBAHAN (5 URL)
  [1] URL    : https://www.google.com/search?q=python+tutorial
      Tokens : ['[CLS]', 'https', ':', '/', '/', 'www', '.', 'google', '.', 'com', '/', 'search', '?', 'q', '=', 'python', '+', 'tutorial', '[SEP]']
      Length : 19 token
  ------------------------------------------------

---
## Section 6 — Otomatisasi Pembaruan File `SKRIPSI_LOG.md`

Menambahkan catatan progres secara otomatis ke bagian paling bawah file `SKRIPSI_LOG.md` di Google Drive. Jika file belum ada, akan dibuat secara otomatis.

In [ ]:
# ============================================================
# 6. Otomatisasi Pembaruan SKRIPSI_LOG.md
# ============================================================
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
date_only = datetime.now().strftime("%Y-%m-%d")

# Ambil nama relatif dataset terhadap PROJECT_DIR untuk log yang bersih
dataset_relpath = os.path.relpath(DATASET_PATH, PROJECT_DIR)

log_entry = f"""

---

## 📝 Log Otomatis — Pelatihan Custom URL Tokenizer (URLBERT)

| Item | Detail |
|---|---|
| **Timestamp** | {timestamp} |
| **Modul / Aktivitas** | Pelatihan Custom WordPiece Tokenizer URL (Metode 1 - URLBERT) |
| **Notebook** | `01_train_custom_url_tokenizer_colab.ipynb` |
| **Lingkungan** | Google Colab |
| **Dataset Masukan** | `{dataset_relpath}` |
| **Total URL Diproses** | {urls_clean_count:,} baris |
| **Baris Dihapus (NaN/kosong)** | {dropped_count:,} baris |
| **Algoritma** | WordPiece (`BertWordPieceTokenizer`) |
| **vocab_size (target)** | {VOCAB_SIZE:,} |
| **Ukuran Kosakata Aktual** | {actual_vocab_size:,} token |
| **Special Tokens** | {', '.join(f'`{t}`' for t in SPECIAL_TOKENS)} |
| **Token `[SEP]`** | Ditiadakan (sesuai rancangan URLBERT — tidak ada ketergantungan antarkalimat) |
| **min_frequency** | {MIN_FREQUENCY} |
| **lowercase** | True |
| **strip_accents** | False |
| **clean_text** | True |
| **Durasi Pelatihan** | {train_duration_str} |
| **Direktori Output** | `{TOKENIZER_OUTPUT_DIR}` |
| **Artefak: vocab.txt** | `{os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.txt')}` |
| **Artefak: tokenizer.json** | `{os.path.join(TOKENIZER_OUTPUT_DIR, 'tokenizer.json')}` |
| **Verifikasi URL Uji** | `{test_url_primary}` → {test_url_token_count} token |
| **Status** | ✅ COMPLETED |
"""

# --- Tulis / Append ke SKRIPSI_LOG.md di Google Drive ---
file_existed = os.path.exists(LOG_PATH)
mode = "a" if file_existed else "w"

with open(LOG_PATH, mode, encoding="utf-8") as f:
    if not file_existed:
        f.write("# 📘 Dynamic Logbook Skripsi\n")
        f.write(f"\n**Terakhir Diperbarui**: {date_only}\n")
    f.write(log_entry)

print(f"[INFO] ✅ Log berhasil {'ditambahkan ke' if file_existed else 'dibuat di'}:")
print(f"  Path: {LOG_PATH}")
print()
print(f"  📋 Ringkasan Log:")
print(f"    Timestamp          : {timestamp}")
print(f"    Dataset            : {dataset_relpath}")
print(f"    Total URL diproses : {urls_clean_count:,}")
print(f"    Vocab size aktual  : {actual_vocab_size:,}")
print(f"    Durasi pelatihan   : {train_duration_str}")
print(f"    URL uji            : {test_url_primary} → {test_url_token_count} token")
print(f"    Output             : {TOKENIZER_OUTPUT_DIR}/")
print()
print("=" * 72)
print("  🎉 SELURUH PROSES SELESAI!")
print("  Tokenizer custom URL (URLBERT) siap digunakan untuk fine-tuning.")
print("=" * 72)